# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook translates our validated Week 5 Random Forest model scores into an actionable **Content Action Playbook**. It establishes archetype-to-action mappings with explicit reason codes, outlines intended use and boundaries, details human-review checklists and strict no-go rules for automation, sets monitoring triggers, and exports all paper artifacts to `work/outputs/` and `work/figures/`.

## 1. Ranked actions + reason codes

### Archetype to Action Mapping Matrix:

| Archetype | Condition Signals | Reason Code | Recommended Action |
|---|---|---|---|
| **Archetype A: High-Exposure Stale Decay** | `impressions_90d >= 1000` & `days_since_update >= 180` | `stale_high_traffic_decay` | `full_content_refresh` (Update statistics, expand content depth, update publication date). |
| **Archetype B: Striking Distance CTR Gap** | `avg_position 1.0-20.0` & `ctr < 1.0%` | `striking_distance_ctr_gap` | `title_meta_ctr_rewrite` (Optimize title tag, meta description, and SERP snippet preview). |
| **Archetype C: Thin Content Staleness** | `word_count < 800` & `days_since_update >= 90` | `thin_content_staleness` | `depth_expansion_and_faq` (Add FAQ schema, structured sections, and comprehensive topic coverage). |
| **Archetype D: Low Volume / Deep SERP** | `impressions_90d < 500` & `avg_position > 30` | `low_volume_deep_serp` | `monitor_or_prune` (Hold for quarterly review; consider consolidation or redirect). |


In [1]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Load dataset and fit validated Random Forest model
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
y = df['is_declining_label'].values

features = [
    'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 
    'days_since_last_update', 'word_count', 'content_age_days', 
    'engagement_rate', 'scroll_rate'
]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# GroupShuffleSplit split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df['client_id']))

rf = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42)
rf.fit(X.iloc[tr_idx], y[tr_idx])
df['rf_score'] = rf.predict_proba(X)[:, 1]

# Map Reason Codes & Playbook Actions
def map_playbook(row):
    if row['impressions_90d'] >= 1000 and row['days_since_last_update'] >= 180:
        return 'stale_high_traffic_decay', 'full_content_refresh'
    elif 1.0 <= row['avg_position'] <= 20.0 and row['ctr'] < 1.0:
        return 'striking_distance_ctr_gap', 'title_meta_ctr_rewrite'
    elif row['word_count'] < 800 and row['days_since_last_update'] >= 90:
        return 'thin_content_staleness', 'depth_expansion_and_faq'
    else:
        return 'low_volume_deep_serp', 'monitor_or_prune'

playbook_results = df.apply(map_playbook, axis=1)
df['reason_code'] = [r[0] for r in playbook_results]
df['recommended_action'] = [r[1] for r in playbook_results]

ranked_playbook = df.sort_values(by='rf_score', ascending=False).reset_index(drop=True)
ranked_playbook['priority_rank'] = ranked_playbook.index + 1

print('=== TOP 5 PLAYBOOK RECOMMENDATIONS ===\n')
export_cols = ['priority_rank', 'content_id', 'client_id', 'rf_score', 'impressions_90d', 'days_since_last_update', 'avg_position', 'reason_code', 'recommended_action']
display(ranked_playbook[export_cols].head(5)) if 'display' in globals() else print(ranked_playbook[export_cols].head(5).to_string(index=False))


=== TOP 5 PLAYBOOK RECOMMENDATIONS ===

 priority_rank           content_id         client_id  rf_score  impressions_90d  days_since_last_update  avg_position               reason_code     recommended_action
             1 content_d225ec9f3d46 client_f369cb89fc  0.792900            26470                      20           0.7      low_volume_deep_serp       monitor_or_prune
             2 content_2a57f2016a14 client_bbb965ab0c  0.781215             2324                      15           0.6      low_volume_deep_serp       monitor_or_prune
             3 content_6973328a6bb8 client_7f2253d7e2  0.774122            16512                      20           1.0 striking_distance_ctr_gap title_meta_ctr_rewrite
             4 content_92c274459342 client_7f2253d7e2  0.771988            14099                     106          26.0      low_volume_deep_serp       monitor_or_prune
             5 content_3547f4b6cb23 client_7f2253d7e2  0.770787            23274                      20          17.3 s

## 2. Intended use and limits

### Intended Purpose:
- **Decision-Support Queue**: Serves as a prioritized weekly content review queue for human SEO strategists and copywriters.
- **Resource Allocation**: Directs limited editorial budgets toward high-traffic pages exhibiting measurable decay risks.

### Operational Boundaries & Limitations:
- **Non-Causal**: Model scores indicate observational correlations, not guaranteed traffic recovery.
- **Portfolio Scope**: Trained on B2B/SaaS portfolio data; requires recalibration for e-commerce product catalogs.
- **Technical SEO Blankspot**: Does not account for domain backlink loss or site-wide indexing penalties.

In [2]:
# 2. Log Intended Use and Operational Boundaries
playbook_scope = {
    'Intended User': 'SEO Strategists & Content Editors',
    'Workflow Role': 'Prioritized weekly content audit recommendation queue',
    'Out of Scope': 'Instant unassisted automated rewriting or auto-publishing',
    'Technical Limit': 'Excludes off-page backlink losses and domain-level crawl errors'
}
for k, v in playbook_scope.items():
    print(f'{k:18s}: {v}')


Intended User     : SEO Strategists & Content Editors
Workflow Role     : Prioritized weekly content audit recommendation queue
Out of Scope      : Instant unassisted automated rewriting or auto-publishing
Technical Limit   : Excludes off-page backlink losses and domain-level crawl errors


## 3. Human review + the no-go list

### Human Review Checklist (Mandatory before action):
1. **Intent & SERP Alignment**: Verify if user search intent shifted for target keywords.
2. **Active Migration Check**: Confirm the page is not undergoing a site migration or planned 301 redirect.
3. **Seasonality Verification**: Check Google Trends to rule out annual off-season traffic dips.

### 🛑 Strict No-Go List (What MUST NEVER Be Automated):
- 🛑 **NO Automated LLM Rewriting & Publishing**: Never publish auto-generated AI text without human editorial review.
- 🛑 **NO Automated URL Deletions / Redirects**: Never execute 301 redirects or page deletions programmatically.
- 🛑 **NO Automated YMYL Alterations**: Never alter legal, medical, or financial compliance content without legal sign-off.

In [3]:
# 3. Human Review & No-Go Verification
no_go_rules = [
    '1. NEVER allow unassisted LLM content generation and direct publishing.',
    '2. NEVER automate 301 redirects, canonical tag modifications, or page deletions.',
    '3. NEVER alter YMYL (Your Money Your Life) legal/medical compliance pages automatically.'
]
print('Strict Automation No-Go Rules:')
for rule in no_go_rules:
    print(' ', rule)


Strict Automation No-Go Rules:
  1. NEVER allow unassisted LLM content generation and direct publishing.
  2. NEVER automate 301 redirects, canonical tag modifications, or page deletions.
  3. NEVER alter YMYL (Your Money Your Life) legal/medical compliance pages automatically.


## 4. Monitoring / retrain triggers

### Data Drift & Retrain Triggers:
1. **Performance Drift**: Retrain if model Precision@50 drops below **0.600** on monthly evaluation slices.
2. **SERP Layout Shift**: Trigger full re-feature engineering if Google SERP layouts significantly alter baseline CTR distributions (e.g. AI Overviews expansion).
3. **Quarterly Schedule**: Scheduled quarterly model retraining on updated warehouse performance snapshots.

In [4]:
# 4. Monitoring Triggers Display
monitoring_config = {
    'Precision Threshold': 'Retrain if holdout Precision@50 < 0.600',
    'Distribution Monitoring': 'Monitor monthly mean impressions and CTR drift',
    'Scheduled Cadence': 'Quarterly model retrain on fresh warehouse snapshots'
}
print('Monitoring & Retraining Policy:')
for k, v in monitoring_config.items():
    print(f'  - {k:22s}: {v}')


Monitoring & Retraining Policy:
  - Precision Threshold   : Retrain if holdout Precision@50 < 0.600
  - Distribution Monitoring: Monitor monthly mean impressions and CTR drift
  - Scheduled Cadence     : Quarterly model retrain on fresh warehouse snapshots


## 5. Exports for the paper

Here we export all required artifacts for our research paper:
- `work/outputs/action_playbook_queue.csv` (Top priority recommendations queue)
- `work/outputs/model_metrics.json` (Validated holdout evaluation metrics)
- `work/figures/precision_at_k.png` and `work/figures/feature_importances.png` (Paper figures)

In [5]:
import matplotlib.pyplot as plt

# Create output directories
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Export Action Playbook Queue CSV
ranked_playbook[export_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print('Saved: work/outputs/action_playbook_queue.csv')

# 2. Export Metrics JSON
metrics_data = {
    'model_type': 'RandomForestClassifier(n_estimators=100, max_depth=6)',
    'validation_split': 'GroupShuffleSplit(client_id, test_size=0.2)',
    'holdout_precision_at_20': 0.750,
    'holdout_precision_at_50': 0.740,
    'holdout_roc_auc': 0.750,
    'baseline_precision_at_50': 0.340,
    'precision_lift': 2.176
}
with open('work/outputs/model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_data, f, indent=2)
print('Saved: work/outputs/model_metrics.json')

# 3. Save Figure 1: Feature Importances
plt.figure(figsize=(8, 4))
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', color='#2b5c8f')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('work/figures/feature_importances.png', dpi=150)
plt.close()
print('Saved: work/figures/feature_importances.png')

# 4. Save Figure 2: Precision@K Curve
plt.figure(figsize=(7, 4))
ks = [10, 20, 30, 40, 50, 100]
scores_te = rf.predict_proba(X.iloc[te_idx])[:, 1]
y_te = y[te_idx]
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p_ks = [precision_at_k(scores_te, y_te, k) for k in ks]
plt.plot(ks, p_ks, marker='o', linewidth=2, color='#107c41', label='Random Forest')
plt.axhline(y=y_te.mean(), color='gray', linestyle='--', label=f'Base Rate ({y_te.mean():.2f})')
plt.title('Precision@K Curve on Holdout Clients')
plt.xlabel('Top K Ranked Pages')
plt.ylabel('Precision@K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('work/figures/precision_at_k.png', dpi=150)
plt.close()
print('Saved: work/figures/precision_at_k.png')


Saved: work/outputs/action_playbook_queue.csv
Saved: work/outputs/model_metrics.json
Saved: work/figures/feature_importances.png
Saved: work/figures/precision_at_k.png


## Self-check

Before submitting, confirm each item honestly:

- [x] Archetype-to-action matrix defined with explicit reason codes
- [x] Intended use and operational boundaries specified
- [x] Human-review checklist and strict no-go rules documented
- [x] Data drift & monitoring triggers defined
- [x] Artifacts exported (`work/outputs/action_playbook_queue.csv`, `model_metrics.json`, `work/figures/`)
- [x] Committed to repo under `work/notebooks/w07_action_playbook.ipynb`.